In [34]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------

import arcpy
import os
import csv
from arcpy import metadata as md



# ---------------------------------------------------------------------
# Input parameters
# ---------------------------------------------------------------------

GeoDB = arcpy.GetParameterAsText(0)

# Optional fallback values if metadata is missing
# DefaultTitle = arcpy.GetParameterAsText(1)
# DefaultTags = arcpy.GetParameterAsText(2)
# DefaultSummary = arcpy.GetParameterAsText(3)
# DefaultDescription = arcpy.GetParameterAsText(4)
# DefaultCredits = arcpy.GetParameterAsText(5)
# DefaultUseLimitations = arcpy.GetParameterAsText(6)
# DefaultContact = arcpy.GetParameterAsText(7)
Delimiter = "|"

# Set workspace
arcpy.env.workspace = GeoDB



# ---------------------------------------------------------------------
# Export Feature Class to JSON
# ---------------------------------------------------------------------

def ExportJSON(FeatureClass):

    try:

        JsonFile = os.path.join(os.path.dirname(arcpy.env.workspace),
                                FeatureClass + ".json")

        if os.path.exists(JsonFile):
            arcpy.AddMessage("File exists: " + JsonFile + ". Deleted")
            os.remove(JsonFile)

        arcpy.AddMessage("Exporting " + JsonFile)

        arcpy.FeaturesToJSON_conversion(
            FeatureClass,
            JsonFile,
            "NOT_FORMATTED"
        )

    except Exception as e:
        arcpy.AddMessage(
            "Export failed for FeatureClass: {} {}".format(
                FeatureClass, str(e)
            )
        )

# ---------------------------------------------------------------------
# Export GeoDB Metadata
# ---------------------------------------------------------------------

def ExportGeoDBMetadata(GeoDB):

    try:

        gdb_meta = md.Metadata(GeoDB)

        GeoDBName = os.path.basename(GeoDB).replace(".gdb", "")

        GDB_CSVPath = os.path.join(
            os.path.dirname(GeoDB),
            GeoDBName + "_metadata.csv"
        )

        if os.path.exists(GDB_CSVPath):
            arcpy.AddMessage("File exists: " + GDB_CSVPath + ". Deleted")
            os.remove(GDB_CSVPath)

        CSVFile = open(GDB_CSVPath, "a", encoding="utf-8")

        CSVFile.write("Geodatabase Metadata\n\n")

        CSVFile.write(f"Title:\n{gdb_meta.title}\n\n")
        CSVFile.write(f"Tags:\n{gdb_meta.tags}\n\n")
        CSVFile.write(f"Summary:\n{gdb_meta.summary}\n\n")
        CSVFile.write(f"Description:\n{gdb_meta.description}\n\n")
        CSVFile.write(f"Credits:\n{gdb_meta.credits}\n\n")
        CSVFile.write(
            f"Use Limitations:\n{gdb_meta.accessConstraints}\n\n"
        )

        CSVFile.close()

    except Exception as e:
        arcpy.AddMessage(
            "GeoDB metadata export failed: " + str(e)
        )



# ---------------------------------------------------------------------
# Export Feature Class to CSV
# ---------------------------------------------------------------------

def ExportCSV(FeatureClass):

    try:

        # Read feature class metadata
        fc_meta = md.Metadata(FeatureClass)

        FCTitle = fc_meta.title
        FCTags = fc_meta.tags
        FCSummary = fc_meta.summary
        FCDescription = fc_meta.description
        FCCredits = fc_meta.credits
        FCUseLimitations = fc_meta.accessConstraints

        FCName = os.path.basename(FeatureClass)

        FC_CSVPath = os.path.join(
            os.path.dirname(arcpy.env.workspace),
            FCName + ".csv"
        )


        if os.path.exists(FC_CSVPath):
            arcpy.AddMessage("File exists: " + FC_CSVPath + ". Deleted")
            os.remove(FC_CSVPath)


        CSVFile = open(
            FC_CSVPath,
            "a",
            encoding="utf-8"
        )


        # Feature class metadata
        CSVFile.write("Feature Class Metadata\n\n")

        CSVFile.write(f"Title:\n{FCTitle}\n\n")
        CSVFile.write(f"Tags:\n{FCTags}\n\n")
        CSVFile.write(f"Summary:\n{FCSummary}\n\n")
        CSVFile.write(f"Description:\n{FCDescription}\n\n")
        CSVFile.write(f"Credits:\n{FCCredits}\n\n")
        CSVFile.write(f"Use Limitations:\n{FCUseLimitations}\n\n")

        spatial_ref = arcpy.Describe(FeatureClass).spatialReference
        CSVFile.write("Spatial Reference:\n"
                      f"Name: {spatial_ref.name}\n"
                      f"WKID: {spatial_ref.factoryCode}\n\n"
                      "Well-Known Text (WKT):\n"
                      f"{spatial_ref.exportToString()}\n\n"
                      )

        CSVFile.write("Purpose:\n"
                      "This file is a human and machine readable equivalent of the layer\n"
                      f">>>{FeatureClass}\n"
                      "exported from the ESRI personal geodatabase\n"
                      f">>>{os.path.basename(GeoDB)}\n"
                      "and was generated to back up and archive the parent dataset\n"
                      "for posterity in a non-proprietary text format.\n\n"
                      )

        CSVFile.write("Note:\n"
                      "Row field values are separated by a pipe character |"
                      "to avoid confusion with commas in WKT geometry.\n\n"
                      )

        fields = arcpy.ListFields(FeatureClass)
        field_names = [field.name for field in fields]
        cursor_fields = ["Shape@WKT"] + field_names

        CSVFile.write(Delimiter.join(cursor_fields) + "\n")

        with arcpy.da.SearchCursor(FeatureClass, cursor_fields) as cursor:
            for row in cursor:
                CSVFile.write(
                    Delimiter.join("" if v is None else str(v) for v in row)
                )
                CSVFile.write("\n")

        CSVFile.close()

    except Exception as e:
        arcpy.AddMessage(
            "Export failed for FeatureClass: {} {}".format(
                FeatureClass, str(e)
            )
        )



# ---------------------------------------------------------------------
# Process all feature classes
# ---------------------------------------------------------------------

# try:

#     ExportGeoDBMetadata(GeoDB)

#     for FeatureClass in arcpy.ListFeatureClasses():

#         arcpy.AddMessage("Processing " + FeatureClass)

#         ExportCSV(FeatureClass)
#         ExportJSON(FeatureClass)

# except Exception as e:

#     arcpy.AddMessage("Error: " + str(e))


ExportGeoDBMetadata(GeoDB)
arcpy.env.workspace = GeoDB

# Root-level feature classes
for RootFC in arcpy.ListFeatureClasses() or []:

    RootFCPath = os.path.join(GeoDB, RootFC)

    arcpy.AddMessage("Processing " + RootFCPath)

    ExportCSV(RootFCPath)
    ExportJSON(RootFCPath)


# Feature classes inside feature datasets
for FeatureDataset in arcpy.ListDatasets(feature_type="Feature") or []:

    for DatasetFC in arcpy.ListFeatureClasses(feature_dataset=FeatureDataset) or []:

        DatasetFCPath = os.path.join(
            GeoDB,
            FeatureDataset,
            DatasetFC
        )

        arcpy.AddMessage("Processing " + DatasetFCPath)

        ExportCSV(DatasetFCPath)
        ExportJSON(DatasetFCPath)

In [35]:
GeoDB = r"Y:/aa_GSS_Student_Engagement_Training/Sho/GeoDBToText/data/multiple_features_test.gdb"

arcpy.env.workspace = GeoDB

ExportGeoDBMetadata(GeoDB)


# Root-level feature classes
for FeatureClass in arcpy.ListFeatureClasses() or []:

    FCPath = os.path.join(GeoDB, FeatureClass)

    arcpy.AddMessage("Processing " + FCPath) 

    ExportCSV(FCPath)
    ExportJSON(FCPath)


# Feature classes inside feature datasets
for FeatureDataset in arcpy.ListDatasets(feature_type="Feature") or []:

    for FeatureClass in arcpy.ListFeatureClasses(feature_dataset=FeatureDataset) or []:

        FCPath = os.path.join(GeoDB, FeatureDataset, FeatureClass)

        arcpy.AddMessage("Processing " + FCPath)

        ExportCSV(FCPath)
        ExportJSON(FCPath)

### double checking if I have access to FC inside Feature dataset

In [15]:
GeoDB = r"Y:/aa_GSS_Student_Engagement_Training/Sho/GeoDBToText/data/multiple_features_test.gdb"

arcpy.env.workspace = GeoDB

print(arcpy.ListDatasets(feature_type="Feature"))

['FeaturDataset']


In [16]:
for fd in arcpy.ListDatasets(feature_type="Feature") or []:
    print("Feature Dataset:", fd)

    print("Feature Classes:", arcpy.ListFeatureClasses(feature_dataset=fd))

Feature Dataset: FeaturDataset
Feature Classes: ['Lakes', 'Rivers']


In [17]:
FCPath = os.path.join(GeoDB, fd, FeatureClass)

print(FCPath)
ExportCSV(FCPath)

Y:/aa_GSS_Student_Engagement_Training/Sho/GeoDBToText/data/multiple_features_test.gdb\FeaturDataset\winona_lake.shp
